Các vấn đề của dữ liệu:
- Metadata (từ VOZ)
- Không chuẩn unicode
- Anh - Việt xen kẽ
- Teencode, từ lóng
- Icon
- Emoji
- Link
- Spam
- Nhiều khoảng trống
- Viết hoa thường không có quy tắc (tránh lộn với case tên riêng)
- Tiếng Việt không dấu
- Cố tính viết sai để lách luật (c.h.e.t, d.m)...
- Kéo dài ký tự (Ây daaaaaaaaaaa)
- Lỗi dính ký tự bộ gõ (ddungws)
- Lỗi lặp từ vô nghĩa
- Câu tỉnh lược
- Lỗi dính từ (alo.Khôi đấy à em)




Giai đoạn 2: Chuẩn hóa cấu trúc (Structural Normalization)
Mục tiêu: Tách bạch các thành phần để máy có thể đọc được đâu là từ, đâu là dấu.

Xử lý Emoji & Icon:

Option 1 (Nên dùng): Thay thế bằng text mô tả (Demojize) -> :smile:, :sad:. Nhớ thêm khoảng trắng 2 đầu.

Option 2: Xóa bỏ (nếu thấy không cần thiết cho cảm xúc).

Lưu ý: Cần làm trước khi xử lý dấu câu để tránh emoji bị dính vào dấu câu.

Xử lý Lỗi dính từ / Dính dấu câu:

Dùng Regex tách các dấu câu bị dính liền với chữ.

Ví dụ: alo.Khôi -> alo . Khôi; chán,không -> chán , không.

Chuẩn hóa khoảng trắng:

Xóa khoảng trắng thừa, tab, xuống dòng, đưa về duy nhất 1 dấu space giữa các từ.

Giai đoạn 3: Chuẩn hóa từ vựng (Lexical Normalization)
Mục tiêu: Đưa các biến thể về dạng từ chuẩn trong từ điển.

Xử lý Kéo dài ký tự (Character Elongation):

Quy tắc: Rút gọn các ký tự lặp lại quá 2 lần về 1 hoặc 2 lần.

Ví dụ: đúuuuung -> đúng, bắtttt -> bắt.

Xử lý "Cố tình viết sai" (Obfuscation) & Teencode cơ bản:

Dùng Regex để clean các dấu chấm giữa từ: c.h.ế.t -> chết.

Map Teencode/Viết tắt thông dụng về từ chuẩn: h -> giờ, k -> không.

Xử lý Lỗi bộ gõ (Typing Errors):

Sửa lỗi ddungws -> đúng. Bước này khá khó, thường dùng thư viện hoặc heuristic rules (quy luật gõ Telex).

Chuẩn hóa dấu thanh:

Đưa về một chuẩn vị trí dấu (ví dụ: hòa thay vì hoà).

Xử lý Tiếng Việt không dấu (Optional):

Nếu dữ liệu không dấu quá nhiều, bạn cần chạy mô hình khôi phục dấu (Accent Restoration) tại đây. Nếu ít, có thể bỏ qua.

Giai đoạn 4: Hoàn thiện (Final Polish)
Chuẩn bị sẵn sàng cho Tokenizer.

Viết thường (Lowercase):

Đưa toàn bộ về chữ thường.

Lưu ý: Với bài toán Hate Speech, tên riêng (Viết Hoa) thường không quá quan trọng, nên lowercase giúp giảm số lượng từ vựng (Vocab size) đáng kể.

Xử lý Lặp từ vô nghĩa:

Loại bỏ các từ lặp lại liên tiếp do lỗi nói lắp (cái cái cái này

In [13]:
import pandas as pd

In [20]:
import glob
raw_data = glob.glob("../data/raw/*.csv")

Gộp dữ liệu

In [48]:
texts = []
for file in raw_data:
    df = pd.read_csv(file)
    cols = df.iloc[:,1]
    cols = cols.dropna()
    cols = cols[cols.str.lower().str.strip() != "text"]
    cols = cols[cols.str.strip() != ""]

    texts.append(cols)


all_texts = pd.concat(texts, ignore_index=True)

combined = pd.DataFrame({'text': all_texts})



In [49]:
combined

,text
0,Thông báo với tất cả các bạn có ý định post bà...
1,URL nick bị ban:\nhttps://voz.vn/u/texdeka.122...
2,Lý do: vi phạm nội quy trong thread CCTV\nMod ...
3,Bạn biết mình vi phạm cụ thể nội quy nào không...
4,URL thread/post bị xóa:\nhttps://voz.vn/t/can-...
...,...
400615,muộn :(((
400616,khà khà giọng cũ đã quay trở lại
400617,Sớm luôn
400618,SỚM GẦN NHẤT


In [50]:
combined.dropna().drop_duplicates()

,text
0,Thông báo với tất cả các bạn có ý định post bà...
1,URL nick bị ban:\nhttps://voz.vn/u/texdeka.122...
2,Lý do: vi phạm nội quy trong thread CCTV\nMod ...
3,Bạn biết mình vi phạm cụ thể nội quy nào không...
4,URL thread/post bị xóa:\nhttps://voz.vn/t/can-...
...,...
400615,muộn :(((
400616,khà khà giọng cũ đã quay trở lại
400617,Sớm luôn
400618,SỚM GẦN NHẤT


Giai đoạn 1: Làm sạch thô (Cleaning)
Mục tiêu: Loại bỏ rác kỹ thuật để giảm nhiễu cho các bước xử lý ngôn ngữ sau.

Xử lý Metadata & Spam: (Ưu tiên số 1)

Loại bỏ các đoạn text thừa của hệ thống VOZ như Sent from..., Quote:, Edit:.

Lọc bỏ các comment spam (trùng lặp hoàn toàn, độ dài quá ngắn hoặc quá dài bất thường).

Xử lý Link & HTML:

Loại bỏ hoặc thay thế URL/Link bằng token đặc biệt (ví dụ <URL>).

Xóa các thẻ HTML còn sót (<br>, &nbsp;).

Chuẩn hóa Unicode (Unicode Normalization):

Đưa toàn bộ về chuẩn Unicode dựng sẵn (NFC). Đây là chuẩn phổ biến nhất hiện nay (khác với tổ hợp NFD trên iOS cũ).

Lý do làm sớm: Để đảm bảo các regex phía sau hoạt động đúng với các ký tự tiếng Việt.